# II - Structuration SQL et suivi de l'activité

## Analyse opérationnelle d'une activité artisanale de desserts

Ce notebook prolonge l'analyse réalisée dans le notebook Python.

Dans la première partie du projet, un modèle théorique de rentabilité a été construit afin d'estimer la performance économique des produits.

Cependant, lors du lancement réel de l'activité, plusieurs écarts sont apparus entre les estimations théoriques et la réalité du terrain.

L'objectif de ce notebook est donc de structurer les données de l'activité avec SQL afin de mieux suivre :

- les produits,
- les ventes,
- les coûts d'approvisionnement,
- l'évolution de l'activité.

Cette approche permet de passer d'une simple estimation de rentabilité à une logique de suivi et de pilotage de l'activité.

## 1. Rappel du modèle théorique

Dans le premier notebook Python, plusieurs indicateurs ont été calculés afin d'estimer la rentabilité des produits.

Ces indicateurs reposaient sur trois informations principales :

- le coût des ingrédients,
- le prix de vente,
- le temps de préparation.

À partir de ces données, plusieurs mesures ont été calculées :

- la marge brute,
- la marge par minute,
- la marge par heure,
- une simulation de production sur une durée donnée.

Cette analyse permettait d'identifier les produits théoriquement les plus rentables.

## 2. Limites du modèle théorique

Lors du lancement réel de l'activité, plusieurs facteurs non anticipés sont apparus.

L'activité étant réalisée seule, plusieurs contraintes organisationnelles ont influencé la production :

- gestion des courses et des approvisionnements,
- préparation des desserts,
- gestion des commandes,
- livraison,
- nettoyage et organisation.

À cela s'ajoutent d'autres facteurs :

- un emploi du temps variable lié aux études,
- une demande fluctuante,
- la gestion des retours clients,
- la pression liée aux délais de préparation,
- la nécessité de s'adapter rapidement.

Ces éléments montrent que la rentabilité réelle dépend aussi de facteurs humains et organisationnels qui ne sont pas toujours pris en compte dans un modèle théorique.

## 3. Décisions prises en situation réelle

Certaines situations imprévues ont nécessité des décisions rapides.

Par exemple, plusieurs pots de pâte de pistache commandés en ligne sont arrivés cassés.

Deux options étaient possibles :

- interrompre temporairement les commandes,
- continuer à accepter les commandes en achetant de la pâte de pistache plus chère en magasin.

La décision prise a été de continuer à prendre des commandes.

Ce choix ne visait pas à maximiser la rentabilité immédiate mais plutôt à :

- maintenir la relation client,
- développer la visibilité de l'activité,
- fidéliser les premiers clients.

De la même manière, les desserts ont été volontairement bien remplis et soignés visuellement afin de donner une bonne image de la marque.

Cela montre que les décisions économiques reposent aussi sur des choix stratégiques et relationnels.

## 4. Objectif de la structuration des données

Face à ces écarts entre théorie et pratique, il devient nécessaire de mieux organiser les données de l'activité.

La mise en place d'une base de données permet de suivre :

- les produits vendus,
- les ventes réalisées,
- les dépenses liées aux ingrédients.

SQL permet alors d'interroger ces données et de produire des analyses utiles pour le pilotage de l'activité.

## 5. Importation des outils

Pour réaliser cette analyse dans JupyterLab, deux outils sont utilisés :

- `sqlite3` pour créer une base de données SQL légère,
- `pandas` pour afficher les résultats des requêtes sous forme de tableaux.

Même si l'objectif est de travailler en SQL, Python permet ici de faire le lien entre la base de données et l'affichage des résultats.

In [4]:
import sqlite3
import pandas as pd

## 6. Création de la base de données

Une base de données SQLite est créée afin de stocker les informations liées à l'activité.

Elle va contenir plusieurs types de données :

- les produits,
- les ventes,
- les approvisionnements.

Cette organisation permet ensuite d'écrire des requêtes SQL pour analyser l'activité.

In [5]:
conn = sqlite3.connect("business_activity.db")
cursor = conn.cursor()

## 7. Création des tables

Trois tables sont créées :

- `produits` : caractéristiques des produits vendus,
- `ventes` : ventes réalisées,
- `approvisionnements` : achats d'ingrédients.

Ces tables permettent de structurer les données de l'activité.

In [6]:
cursor.executescript("""
DROP TABLE IF EXISTS ventes;
DROP TABLE IF EXISTS produits;
DROP TABLE IF EXISTS approvisionnements;

CREATE TABLE produits (
    id_produit INTEGER PRIMARY KEY,
    nom_produit TEXT,
    categorie TEXT,
    temps_preparation_min INTEGER,
    cout_unitaire REAL,
    prix_vente REAL
);

CREATE TABLE ventes (
    id_vente INTEGER PRIMARY KEY,
    date_vente TEXT,
    id_produit INTEGER,
    quantite INTEGER,
    prix_unitaire_vente REAL
);

CREATE TABLE approvisionnements (
    id_appro INTEGER PRIMARY KEY,
    date_appro TEXT,
    nom_ingredient TEXT,
    quantite_achetee REAL,
    cout_total REAL,
    fournisseur TEXT
);
""")

conn.commit()

## 8. Insertion des données

Des données cohérentes avec l'activité sont insérées afin de simuler :

- les produits proposés,
- plusieurs ventes,
- quelques achats d'ingrédients.

L'objectif est de construire une base simple mais exploitable pour les analyses SQL.

In [8]:
cursor.executemany("""
INSERT INTO produits VALUES (?,?,?,?,?,?)
""", [
    (1, "Strawberry Crunch Cup moyenne", "cup", 10, 2.70, 6.00),
    (2, "Strawberry Crunch Cup grande", "cup", 10, 3.85, 8.00),
    (3, "Cookie Crunch Cup", "cookie", 20, 1.90, 6.00),
    (4, "Cookie Cup", "cookie", 15, 1.40, 4.00),
    (5, "Box Découverte", "box", 40, 7.30, 13.00)
])

In [9]:
cursor.executemany("""
INSERT INTO ventes VALUES (?,?,?,?,?)
""", [
    (1, "2026-03-01", 2, 3, 8.00),
    (2, "2026-03-01", 3, 2, 6.00),
    (3, "2026-03-01", 4, 4, 4.00),
    (4, "2026-03-02", 2, 2, 8.00),
    (5, "2026-03-02", 1, 3, 6.00),
    (6, "2026-03-02", 5, 1, 13.00),
    (7, "2026-03-03", 3, 3, 6.00),
    (8, "2026-03-03", 4, 2, 4.00),
    (9, "2026-03-03", 2, 4, 8.00),
    (10, "2026-03-04", 1, 2, 6.00),
    (11, "2026-03-04", 5, 1, 13.00),
    (12, "2026-03-04", 3, 1, 6.00)
])

In [10]:
cursor.executemany("""
INSERT INTO approvisionnements VALUES (?,?,?,?,?,?)
""", [
    (1, "2026-03-01", "Spéculoos biscuits", 1.0, 13.0, "Carrefour"),
    (2, "2026-03-01", "Pâte de pistache", 0.6, 18.0, "Amazon"),
    (3, "2026-03-01", "Nutella", 1.0, 7.0, "Carrefour"),
    (4, "2026-03-01", "Crème chocolat blanc", 1.0, 10.0, "Amazon"),
    (5, "2026-03-01", "Corn flakes", 1.0, 3.0, "Carrefour"),
    (6, "2026-03-01", "Spéculoos en poudre", 1.0, 8.0, "Atacadao"),
    (7, "2026-03-01", "Pistaches fraîches", 1.0, 16.0, "Carrefour"),
    (8, "2026-03-01", "Kinder Bueno", 1.0, 12.0, "Carrefour"),
    (9, "2026-03-01", "Fraises", 1.0, 9.0, "Marché local"),
    (10, "2026-03-01", "Farine", 1.0, 2.0, "Carrefour"),
    (11, "2026-03-01", "Chocolat lait", 1.0, 8.0, "Carrefour"),
    (12, "2026-03-01", "Chocolat blanc", 1.0, 9.0, "Carrefour"),
    (13, "2026-03-01", "Vergeoise", 1.0, 3.0, "Carrefour"),
    (14, "2026-03-01", "Œufs", 1.0, 5.0, "Carrefour"),
    (15, "2026-03-01", "Beurre", 1.0, 9.0, "Carrefour")
])

In [11]:
conn.commit()

## 9. Question d'analyse

Afin de mieux comprendre l'activité, plusieurs questions simples peuvent être posées :

- Quels produits génèrent le plus de chiffre d'affaires ?
- Quels produits se vendent le plus ?
- Quels produits génèrent le plus de marge ?
- Quels fournisseurs représentent les dépenses les plus importantes ?

Les requêtes SQL suivantes permettent de répondre à ces questions.

## 10. Vérification de la table des produits

Avant de commencer les analyses, il est utile de vérifier que les produits ont bien été enregistrés.

In [12]:
query = """
SELECT *
FROM produits;
"""

pd.read_sql_query(query, conn)

,id_produit,nom_produit,categorie,temps_preparation_min,cout_unitaire,prix_vente
0,1,Strawberry Crunch Cup moyenne,cup,10,2.70,6.0
1,2,Strawberry Crunch Cup grande,cup,10,3.85,8.0
2,3,Cookie Crunch Cup,cookie,20,1.90,6.0
3,4,Cookie Cup,cookie,15,1.40,4.0
4,5,Box Découverte,box,40,7.30,13.0


## 11. Chiffre d'affaires par produit

Cette requête permet d'identifier quels produits génèrent le plus de chiffre d'affaires.

In [13]:
query = """
SELECT p.nom_produit,
       SUM(v.quantite * v.prix_unitaire_vente) AS chiffre_affaires
FROM ventes v
JOIN produits p ON v.id_produit = p.id_produit
GROUP BY p.nom_produit
ORDER BY chiffre_affaires DESC;
"""

pd.read_sql_query(query, conn)

,nom_produit,chiffre_affaires
0,Strawberry Crunch Cup grande,72.0
1,Cookie Crunch Cup,36.0
2,Strawberry Crunch Cup moyenne,30.0
3,Box Découverte,26.0
4,Cookie Cup,24.0


## 12. Quantités vendues par produit

Cette requête permet d'identifier les produits les plus demandés.

In [14]:
query = """
SELECT p.nom_produit,
       SUM(v.quantite) AS quantite_vendue
FROM ventes v
JOIN produits p ON v.id_produit = p.id_produit
GROUP BY p.nom_produit
ORDER BY quantite_vendue DESC;
"""

pd.read_sql_query(query, conn)

,nom_produit,quantite_vendue
0,Strawberry Crunch Cup grande,9
1,Cookie Cup,6
2,Cookie Crunch Cup,6
3,Strawberry Crunch Cup moyenne,5
4,Box Découverte,2


## 13. Marge totale par produit

Cette requête permet d'estimer quels produits génèrent le plus de marge.

In [15]:
query = """
SELECT p.nom_produit,
       SUM((v.prix_unitaire_vente - p.cout_unitaire) * v.quantite) AS marge_totale
FROM ventes v
JOIN produits p ON v.id_produit = p.id_produit
GROUP BY p.nom_produit
ORDER BY marge_totale DESC;
"""

pd.read_sql_query(query, conn)

,nom_produit,marge_totale
0,Strawberry Crunch Cup grande,37.35
1,Cookie Crunch Cup,24.60
2,Strawberry Crunch Cup moyenne,16.50
3,Cookie Cup,15.60
4,Box Découverte,11.40


## 14. Dépenses par fournisseur

Cette requête permet d'identifier les fournisseurs qui représentent les coûts les plus importants.

In [16]:
query = """
SELECT fournisseur,
       SUM(cout_total) AS depense_totale
FROM approvisionnements
GROUP BY fournisseur
ORDER BY depense_totale DESC;
"""

pd.read_sql_query(query, conn)

,fournisseur,depense_totale
0,Carrefour,87.0
1,Amazon,28.0
2,Marché local,9.0
3,Atacadao,8.0


## 15. Coût par ingrédient

Cette requête permet de comparer les ingrédients selon leur coût d'achat.

Elle permet d'identifier les matières premières les plus sensibles pour la rentabilité.

In [17]:
query = """
SELECT nom_ingredient,
       fournisseur,
       ROUND(cout_total / quantite_achetee, 2) AS cout_par_unite
FROM approvisionnements
ORDER BY cout_par_unite DESC;
"""

pd.read_sql_query(query, conn)

,nom_ingredient,fournisseur,cout_par_unite
0,Pâte de pistache,Amazon,30.0
1,Pistaches fraîches,Carrefour,16.0
2,Spéculoos biscuits,Carrefour,13.0
3,Kinder Bueno,Carrefour,12.0
4,Crème chocolat blanc,Amazon,10.0
5,Fraises,Marché local,9.0
6,Chocolat blanc,Carrefour,9.0
7,Beurre,Carrefour,9.0
8,Spéculoos en poudre,Atacadao,8.0
9,Chocolat lait,Carrefour,8.0


## 16. Limites de l'analyse

Les données utilisées dans ce notebook sont simplifiées et ne représentent pas l'ensemble des facteurs influençant l'activité.

En réalité, plusieurs éléments peuvent également avoir un impact :

- le temps disponible pour la production,
- la gestion des stocks,
- la relation client,
- la variabilité de la demande,
- les imprévus logistiques.

Ces éléments montrent que l'analyse des données doit toujours être interprétée dans son contexte opérationnel.

## Conclusion

L'analyse Python permettait de simuler la rentabilité théorique.

L'expérience réelle a montré que la gestion d'une activité dépend aussi :

- de facteurs humains,
- de contraintes organisationnelles,
- de décisions stratégiques.

La structuration des données avec SQL permet donc de mieux suivre l'activité et de produire des analyses utiles pour la prise de décision.

Le projet montre ainsi que les chiffres sont essentiels, mais qu'ils doivent toujours être interprétés avec les réalités du terrain.

In [18]:
conn.close()